In [1]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

# ----------------------------
# Clinical metadata, GSE114725 (Azizi et al. 2018, Figure S1A) —
# transcribed directly from the paper's supplementary figure.
# CORRECTED: Her2 status is binary (-/+), only BC7 is HER2+; earlier
# transcription incorrectly introduced an intermediate "-+" category.
# ----------------------------
clinical_data = pd.DataFrame({
    "patient": ["BC1", "BC2", "BC3", "BC4", "BC5", "BC6", "BC7", "BC8"],
    "size_cm": [1, 3, 1.5, 2.1, 2, 1.3, 1.2, 1.3],
    "metastases": [0, 1, 0, 1, 0, 0, 0, 0],
    "grade": [1, 2, 3, 1, 3, 2, 3, 2],
    "ER": [0.95, 0.9, 0.0, 0.95, 0.05, 0.99, 0.0, 0.2],
    "PR": [0.95, 0.1, 0.0, 0.95, 0.01, 0.01, 0.0, 0.05],
    "Her2": ["-", "-", "-", "-", "-", "-", "+", "-"],
    "post_menopause": ["-", "+", "-", "-", "+", "+", "+", "+"],
    "age": [38, 60, 43, 52, 78, 58, 65, 72],
    "subtype": ["Ductal"] * 8,
    "BRCA_deficiency": ["Negative", "Unknown", "Negative", "Negative", "Unknown", "Unknown", "Unknown", "Unknown"],
})

print(clinical_data)
clinical_data.to_csv(RESULTS_DIR / "GSE114725_patient_clinical_metadata.csv", index=False)
print("\nSaved clinical metadata table")

  patient  size_cm  metastases  grade    ER    PR Her2 post_menopause  age  \
0     BC1      1.0           0      1  0.95  0.95    -              -   38   
1     BC2      3.0           1      2  0.90  0.10    -              +   60   
2     BC3      1.5           0      3  0.00  0.00    -              -   43   
3     BC4      2.1           1      1  0.95  0.95    -              -   52   
4     BC5      2.0           0      3  0.05  0.01    -              +   78   
5     BC6      1.3           0      2  0.99  0.01    -              +   58   
6     BC7      1.2           0      3  0.00  0.00    +              +   65   
7     BC8      1.3           0      2  0.20  0.05    -              +   72   

  subtype BRCA_deficiency  
0  Ductal        Negative  
1  Ductal         Unknown  
2  Ductal        Negative  
3  Ductal        Negative  
4  Ductal         Unknown  
5  Ductal         Unknown  
6  Ductal         Unknown  
7  Ductal         Unknown  

Saved clinical metadata table


In [2]:
import scanpy as sc
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")

# Restrict to Tumour tissue only — confound-free comparison
tumor_only = adata1.obs[adata1.obs["tissue"] == "TUMOR"].copy()

# Total cells per patient, Tumour only
patient_totals_tumor = tumor_only.groupby("patient", observed=True).size()
print("Tumour-only cell counts per patient:")
print(patient_totals_tumor)

# Total macrophage/monocyte % per patient, Tumour only
mac_mask = tumor_only["cell_type_fine"].str.contains("macrophage|monocyte", case=False, na=False)
mac_counts_tumor = tumor_only[mac_mask].groupby("patient", observed=True).size()
mac_pct_tumor = (mac_counts_tumor / patient_totals_tumor * 100).reindex(patient_totals_tumor.index, fill_value=0)

print("\n=== Total macrophage/monocyte %, TUMOUR TISSUE ONLY ===")
print(mac_pct_tumor.sort_values(ascending=False))

# Highlight metastatic patients
metastatic = ["BC2", "BC4"]
print(f"\nMetastatic patients (BC2, BC4): {mac_pct_tumor[metastatic].to_dict()}")
print(f"Non-metastatic patients mean: {mac_pct_tumor.drop(metastatic).mean():.2f}%")

Tumour-only cell counts per patient:
patient
BC1    1366
BC2    1912
BC3     647
BC4    6133
BC5    1764
BC6    3497
BC7    1679
BC8    2596
dtype: int64

=== Total macrophage/monocyte %, TUMOUR TISSUE ONLY ===
patient
BC3    63.678516
BC6    53.217043
BC8    39.098613
BC7    35.616438
BC5    25.226757
BC1    17.057101
BC4     8.250448
BC2     4.654812
dtype: float64

Metastatic patients (BC2, BC4): {'BC2': 4.654811715481171, 'BC4': 8.250448393934454}
Non-metastatic patients mean: 38.98%


In [1]:
import scanpy as sc
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"

adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")
print(adata1.obs.groupby(["patient", "tissue"], observed=True).size().unstack(fill_value=0))

tissue   BLOOD  LYMPHNODE  NORMAL  TUMOR
patient                                 
BC1       2736          0    2750   1366
BC2          0       5136    1157   1912
BC3          0          0     249    647
BC4      12856          0       0   6133
BC5          0          0       0   1764
BC6          0          0       0   3497
BC7          0          0     184   1679
BC8          0          0       0   2596


In [3]:
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

clinical_data = pd.read_csv(RESULTS_DIR / "GSE114725_patient_clinical_metadata.csv").set_index("patient")

# ----------------------------
# Wider exploration — metastatic (BC2, BC4) vs non-metastatic patients,
# across overall cell-type composition, not just macrophages. Casting
# a broad net first, per Adrien's open-ended framing, before narrowing
# in on any specific population that looks interesting.
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")

metastatic_patients = ["BC2", "BC4"]
non_metastatic_patients = [p for p in clinical_data.index if p not in metastatic_patients]

# Per-patient composition across ALL cell_type_fine categories
comp = adata1.obs.groupby(["patient", "cell_type_fine"], observed=True).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

print("Cell-type composition (%), metastatic patients:")
print(comp_pct.loc[metastatic_patients].T)
print("\nCell-type composition (%), non-metastatic patients (mean):")
print(comp_pct.loc[non_metastatic_patients].mean().sort_values(ascending=False))

print("\n=== Comparing metastatic vs non-metastatic, per cell type ===")
results = []
for ct in comp_pct.columns:
    met_vals = comp_pct.loc[metastatic_patients, ct]
    nonmet_vals = comp_pct.loc[non_metastatic_patients, ct]
    met_mean = met_vals.mean()
    nonmet_mean = nonmet_vals.mean()
    diff = met_mean - nonmet_mean
    results.append({"cell_type_fine": ct, "metastatic_mean_pct": met_mean,
                     "nonmetastatic_mean_pct": nonmet_mean, "difference": diff})

results_df = pd.DataFrame(results).sort_values("difference", key=abs, ascending=False)
print(results_df.head(15).to_string(index=False))

Cell-type composition (%), metastatic patients:
patient                                                   BC2        BC4
cell_type_fine                                                          
Activated CD8 T cells                               12.577756   8.356576
Antigen-presenting macrophages                       0.398109   0.442986
B cells                                             15.874596   7.038556
CD4 Activated T cells                               36.501617  13.601313
CD4 Naive/Resting T cells                           10.674297  28.044846
Complement-high macrophages                          0.497636   0.596117
Cycling CD8 T cells                                  0.174173   0.131255
Effector CD8 T cells                                 3.159990   5.354115
LAM-like macrophages                                 0.298582   0.202352
Lipid-laden/Foam-cell macrophages                    0.062205   0.082034
Mast cells                                           0.161732   1.274269
Mix

In [4]:
# ----------------------------
# Check for confounding: are BC2/BC4 unusual on other clinical
# variables too, or specifically distinguished by metastasis?
# ----------------------------
print("Clinical profile of all 8 patients, for comparison:")
print(clinical_data[["grade", "ER", "PR", "Her2", "metastases"]])

# Total macrophage % per patient (summed across all 8 macrophage
# sub-types), to see the pattern as one number per patient rather
# than split across sub-types
mac_cols = [c for c in comp_pct.columns if "macrophage" in c.lower() or "monocyte" in c.lower()]
total_mac_pct = comp_pct[mac_cols].sum(axis=1)
print("\nTotal macrophage/monocyte %, all patients (sorted):")
print(total_mac_pct.sort_values(ascending=False))

Clinical profile of all 8 patients, for comparison:
         grade    ER    PR Her2  metastases
patient                                    
BC1          1  0.95  0.95    -           0
BC2          2  0.90  0.10    -           1
BC3          3  0.00  0.00    -           0
BC4          1  0.95  0.95    -           1
BC5          3  0.05  0.01    -           0
BC6          2  0.99  0.01    -           0
BC7          3  0.00  0.00    +           0
BC8          2  0.20  0.05    -           0

Total macrophage/monocyte %, all patients (sorted):
patient
BC3    67.358708
BC6    53.584797
BC8    39.295393
BC7    36.917170
BC5    25.648415
BC1    11.471990
BC4     3.664206
BC2     2.040309
dtype: float64


In [5]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

# ----------------------------
# Clinical metadata, GSE176078 (Wu et al. 2021 supplementary table).
# Case IDs mapped to actual sample IDs (hyphenated numeric IDs in the
# paper correspond to CID-prefixed IDs with letter/number suffixes in
# the dataset, confirmed by direct match against known 26-sample cohort).
# NOTE: Ki67 kept as raw text — source table mixes percentage ranges
# and plain decimals inconsistently; needs manual clarification before
# use in analysis.
# ----------------------------
clinical_data_2 = pd.DataFrame([
    ["CID3586", 43, 3, "IDC", "HER2+/ER+", "Naive", "-", "30-50%"],
    ["CID3838", 49, 3, "IDC", "HER2+", "Naive", "-", "0.6"],
    ["CID3921", 60, 3, "IDC", "HER2+", "Naive", "-", ">50%"],
    ["CID3941", 50, 2, "IDC", "ER+", "Naive", "-", "0.1"],
    ["CID3946", 52, 3, "IDC", "TNBC", "Naive", "-", "0.6"],
    ["CID3948", 82, 3, "IDC", "ER+", "Naive", "-", "~10%"],
    ["CID3963", 61, 3, "IDC", "ER+", "Treated", "AC, Paclitaxel, Herceptin (3yr prior)", "0.43"],
    ["CID4040", 57, 3, "IDC", "ER+", "Naive", "-", ">50%"],
    ["CID4066", 41, 2, "IDC", "HER2+/ER+", "Treated", "Neoadjuvant AC", "0.3"],
    ["CID4067", 85, 2, "IDC", "ER+", "Naive", "-", "3-4%"],
    ["CID4290A", 88, 2, "IDC", "ER+", "Naive", "-", "0.1"],
    ["CID4398", 52, 3, "IDC", "ER+", "Treated", "Neoadjuvant FEC-D", "0.75"],
    ["CID44041", 35, 3, "IDC", "TNBC", "Naive", "-", "0.7"],
    ["CID4461", 54, 2, "IDC", "ER+", "Naive", "-", "0.15"],
    ["CID4463", 58, 2, "IDC", "ER+", "Naive", "-", "0.5"],
    ["CID4465", 54, 3, "IDC", "TNBC", "Naive", "-", "0.7"],
    ["CID4471", 55, 2, "ILC", "ER+", "Naive", "-", "0.2"],
    ["CID4495", 63, 3, "IDC", "TNBC", "Naive", "-", "0.8"],
    ["CID44971", 49, 3, "IDC", "TNBC", "Naive", "-", "0.4"],
    ["CID44991", 47, 3, "IDC", "TNBC", "Naive", "-", "60-70%"],
    ["CID4513", 73, 3, "MBC", "TNBC", "Treated", "Neoadjuvant AC (4x), Paclitaxel (3x)", "0.75"],
    ["CID4515", 67, 3, "IDC", "TNBC", "Naive", "-", "0.6"],
    ["CID45171", 58, 3, "IDC", "HER2+", "Naive", "-", "0.8"],
    ["CID4523", 52, 3, "MBC", "TNBC", "Treated", "Neoadjuvant AC (4x), Paclitaxel (1x)", "0.9"],
    ["CID4530N", 42, 2, "IDC", "ER+", "Naive", "-", "0.05"],
    ["CID4535", 47, 2, "ILC", "ER+", "Naive", "-", "0.1"],
], columns=["patient", "age", "grade", "cancer_type", "subtype_ihc", "treatment_status", "treatment_details", "ki67_raw"])

print(clinical_data_2)
print(f"\nTotal patients: {len(clinical_data_2)}")
print(f"\nTreatment status counts:")
print(clinical_data_2["treatment_status"].value_counts())

clinical_data_2.to_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv", index=False)
print("\nSaved clinical metadata table")

     patient  age  grade cancer_type subtype_ihc treatment_status  \
0    CID3586   43      3         IDC   HER2+/ER+            Naive   
1    CID3838   49      3         IDC       HER2+            Naive   
2    CID3921   60      3         IDC       HER2+            Naive   
3    CID3941   50      2         IDC         ER+            Naive   
4    CID3946   52      3         IDC        TNBC            Naive   
5    CID3948   82      3         IDC         ER+            Naive   
6    CID3963   61      3         IDC         ER+          Treated   
7    CID4040   57      3         IDC         ER+            Naive   
8    CID4066   41      2         IDC   HER2+/ER+          Treated   
9    CID4067   85      2         IDC         ER+            Naive   
10  CID4290A   88      2         IDC         ER+            Naive   
11   CID4398   52      3         IDC         ER+          Treated   
12  CID44041   35      3         IDC        TNBC            Naive   
13   CID4461   54      2         I

In [6]:
import scanpy as sc
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

clinical_data_2 = pd.read_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv").set_index("patient")

# ----------------------------
# Treated vs Naive — wide-net composition comparison, same approach
# as the GSE114725 metastasis check. Using cell_type (parent level,
# full lineage breadth) given this is a broad, exploratory first pass.
# Loading metadata only (backed mode) — composition doesn't need the
# full expression matrix, avoids GSE176078's memory sensitivity.
# ----------------------------
adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
obs2 = adata2_meta.obs[["orig.ident", "cell_type"]].copy()
del adata2_meta

treated_patients = clinical_data_2[clinical_data_2["treatment_status"] == "Treated"].index.tolist()
naive_patients = clinical_data_2[clinical_data_2["treatment_status"] == "Naive"].index.tolist()
print(f"Treated: {treated_patients}")
print(f"Naive: {len(naive_patients)} patients")

comp = obs2.groupby(["orig.ident", "cell_type"], observed=True).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

results = []
for ct in comp_pct.columns:
    treated_mean = comp_pct.loc[treated_patients, ct].mean()
    naive_mean = comp_pct.loc[naive_patients, ct].mean()
    diff = treated_mean - naive_mean
    results.append({"cell_type": ct, "treated_mean_pct": treated_mean,
                     "naive_mean_pct": naive_mean, "difference": diff})

results_df = pd.DataFrame(results).sort_values("difference", key=abs, ascending=False)
print("\n=== Treated vs Naive, per cell type ===")
print(results_df.to_string(index=False))

Treated: ['CID3963', 'CID4066', 'CID4398', 'CID4513', 'CID4523']
Naive: 21 patients

=== Treated vs Naive, per cell type ===
                                         cell_type  treated_mean_pct  naive_mean_pct  difference
                                Luminal epithelial          9.946776       25.794458  -15.847683
                                       Macrophages         17.529672        7.692722    9.836950
                                       CD8 T cells         16.465008       10.234174    6.230834
                                 Endothelial cells          4.059318        8.403646   -4.344328
                                               PVL          3.866751        7.820677   -3.953926
                                          NK cells          6.751275        2.863054    3.888220
                                           B cells          0.299454        3.278133   -2.978679
                                    Memory T cells         11.827103        9.104327    2.722777
  

In [7]:
import scanpy as sc
import anndata as ad
import pandas as pd
from pathlib import Path
from sccoda.util import comp_ana as scc_ana

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

clinical_data_2 = pd.read_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv").set_index("patient")

adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
obs2 = adata2_meta.obs[["orig.ident", "cell_type"]].copy()
del adata2_meta

# ----------------------------
# scCODA: Treated vs Naive — properly accounts for the compositional
# (sum-to-100%) structure, unlike the simple descriptive comparison
# above. Same reference cell type (PVL) as prior GSE176078 scCODA runs
# for methodological consistency.
# ----------------------------
comp_df = obs2.groupby(["orig.ident", "cell_type"], observed=True).size().unstack(fill_value=0)
comp_df["treatment_status"] = clinical_data_2.loc[comp_df.index, "treatment_status"].values

sccoda_data = ad.AnnData(
    X=comp_df.drop(columns=["treatment_status"]).values.astype(float),
    obs=comp_df[["treatment_status"]].reset_index().rename(columns={"orig.ident": "sample"}),
    var=pd.DataFrame(index=comp_df.drop(columns=["treatment_status"]).columns)
)
sccoda_data.obs["treatment_status"] = sccoda_data.obs["treatment_status"].astype(str)

model = scc_ana.CompositionalAnalysis(
    sccoda_data, formula="treatment_status", reference_cell_type="PVL"
)
result = model.sample_hmc()
print("=== GSE176078: Treated vs Naive — scCODA results ===")
result.summary()

credible = result.credible_effects()
print("\nCredible effects:")
print(credible)

result.effect_df.to_csv(RESULTS_DIR / "GSE176078_scCODA_treatment_status_effects.csv")
print("\nSaved")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [03:37<00:00, 91.84it/s] 


MCMC sampling finished. (276.654 sec)
Acceptance rate: 51.6%
=== GSE176078: Treated vs Naive — scCODA results ===
Compositional Analysis summary:

Data: 26 samples, 17 cell types
Reference index: 12
Formula: treatment_status

Intercepts:
                                                    Final Parameter  \
Cell Type                                                             
B cells                                                      -0.897   
Basal epithelial                                             -1.427   
CAFs                                                         -0.182   
CD8 T cells                                                   0.079   
Cycling T cells                                              -1.161   
Cycling epithelial                                           -0.805   
Endothelial cells                                            -0.095   
Epithelial (ambiguous)                                       -0.776   
Luminal epithelial                                  

In [8]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"
clinical_data_2 = pd.read_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv").set_index("patient")

# ----------------------------
# Quick group-size check before committing to full scCODA runs
# (each run takes ~4-5 min) — confirms which variables have enough
# samples per group to be worth testing.
# ----------------------------
print("Grade:")
print(clinical_data_2["grade"].value_counts())

print("\nCancer type:")
print(clinical_data_2["cancer_type"].value_counts())

print("\nAge — summary stats (for median split):")
print(clinical_data_2["age"].describe())
median_age = clinical_data_2["age"].median()
print(f"\nMedian age: {median_age}")
print(f"Above/below median split:")
print((clinical_data_2["age"] > median_age).value_counts())

Grade:
grade
3    17
2     9
Name: count, dtype: int64

Cancer type:
cancer_type
IDC    22
ILC     2
MBC     2
Name: count, dtype: int64

Age — summary stats (for median split):
count    26.000000
mean     56.692308
std      13.298930
min      35.000000
25%      49.000000
50%      54.000000
75%      60.750000
max      88.000000
Name: age, dtype: float64

Median age: 54.0
Above/below median split:
age
False    14
True     12
Name: count, dtype: int64


In [9]:
import scanpy as sc
import anndata as ad
import pandas as pd
from pathlib import Path
from sccoda.util import comp_ana as scc_ana

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

clinical_data_2 = pd.read_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv").set_index("patient")
median_age = clinical_data_2["age"].median()
clinical_data_2["age_group"] = (clinical_data_2["age"] > median_age).map({True: "Older", False: "Younger"})
clinical_data_2["grade_group"] = clinical_data_2["grade"].map({2: "Grade2", 3: "Grade3"})

adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
obs2 = adata2_meta.obs[["orig.ident", "cell_type"]].copy()
del adata2_meta

comp_df = obs2.groupby(["orig.ident", "cell_type"], observed=True).size().unstack(fill_value=0)

def run_sccoda_covariate(comp_df, clinical_data, covariate_col, reference="PVL"):
    df = comp_df.copy()
    df[covariate_col] = clinical_data.loc[df.index, covariate_col].values
    cell_cols = [c for c in df.columns if c != covariate_col]
    sccoda_data = ad.AnnData(
        X=df[cell_cols].values.astype(float),
        obs=df[[covariate_col]].reset_index().rename(columns={"orig.ident": "sample"}),
        var=pd.DataFrame(index=cell_cols)
    )
    sccoda_data.obs[covariate_col] = sccoda_data.obs[covariate_col].astype(str)
    model = scc_ana.CompositionalAnalysis(sccoda_data, formula=covariate_col, reference_cell_type=reference)
    result = model.sample_hmc()
    return result

print("=== GRADE (Grade2 vs Grade3) ===")
result_grade = run_sccoda_covariate(comp_df, clinical_data_2, "grade_group")
result_grade.summary()
print("\nCredible effects:")
print(result_grade.credible_effects())
result_grade.effect_df.to_csv(RESULTS_DIR / "GSE176078_scCODA_grade_effects.csv")

print("\n\n=== AGE (Older vs Younger, median split) ===")
result_age = run_sccoda_covariate(comp_df, clinical_data_2, "age_group")
result_age.summary()
print("\nCredible effects:")
print(result_age.credible_effects())
result_age.effect_df.to_csv(RESULTS_DIR / "GSE176078_scCODA_age_effects.csv")

print("\n\nBoth complete")

=== GRADE (Grade2 vs Grade3) ===
Zero counts encountered in data! Added a pseudocount of 0.5.


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [03:42<00:00, 90.05it/s] 


MCMC sampling finished. (278.604 sec)
Acceptance rate: 44.4%
Compositional Analysis summary:

Data: 26 samples, 17 cell types
Reference index: 12
Formula: grade_group

Intercepts:
                                                    Final Parameter  \
Cell Type                                                             
B cells                                                      -0.865   
Basal epithelial                                             -1.307   
CAFs                                                          0.024   
CD8 T cells                                                   0.197   
Cycling T cells                                              -1.148   
Cycling epithelial                                           -0.659   
Endothelial cells                                             0.802   
Epithelial (ambiguous)                                       -0.608   
Luminal epithelial                                            2.004   
Macrophages                            

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
100%|██████████| 20000/20000 [03:36<00:00, 92.51it/s] 


MCMC sampling finished. (270.131 sec)
Acceptance rate: 46.3%
Compositional Analysis summary:

Data: 26 samples, 17 cell types
Reference index: 12
Formula: age_group

Intercepts:
                                                    Final Parameter  \
Cell Type                                                             
B cells                                                      -0.923   
Basal epithelial                                             -1.443   
CAFs                                                         -0.197   
CD8 T cells                                                   0.093   
Cycling T cells                                              -1.152   
Cycling epithelial                                           -0.798   
Endothelial cells                                            -0.125   
Epithelial (ambiguous)                                       -0.777   
Luminal epithelial                                            0.221   
Macrophages                              

In [11]:
import scanpy as sc
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase2_clustering_v2"

clinical_data_2 = pd.read_csv(RESULTS_DIR / "GSE176078_patient_clinical_metadata.csv").set_index("patient")

adata2_meta = sc.read_h5ad(
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r"
)
subtype_lookup = adata2_meta.obs.drop_duplicates("orig.ident").set_index("orig.ident")["subtype"]
del adata2_meta

clinical_data_2["subtype"] = subtype_lookup.loc[clinical_data_2.index]

print(pd.crosstab(clinical_data_2["grade"], clinical_data_2["subtype"]))

subtype  ER+  HER2+  TNBC
grade                    
2          8      1     0
3          3      4    10
